<a href="https://colab.research.google.com/github/Mandeep8252/AI-LAB/blob/main/Week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# First Order Logic - Forward Chaining

# Knowledge Base (Initial Facts)
KB = [
    ("American", "Robert"),
    ("Missile", "T1"),
    ("Owns", "Robert", "T1"),
    ("Sells", "Robert", "T1", "A"),
    ("Enemy", "America", "A")
]

# Inference Rules
rules = [
    # Missiles are weapons
    ([("Missile", "?x")],
     ("Weapon", "?x")),

    # Enemies of America are hostile
    ([("Enemy", "America", "?x")],
     ("Hostile", "?x")),

    # Americans selling weapons to hostile nations are criminals
    ([
        ("American", "?x"),
        ("Weapon", "?y"),
        ("Sells", "?x", "?y", "?z"),
        ("Hostile", "?z")
    ],
     ("Criminal", "?x"))
]


# Match two expressions and find substitutions
def unify(pattern, fact, subst):

    if len(pattern) != len(fact):
        return None

    for p, f in zip(pattern, fact):

        if p.startswith("?"):

            if p in subst:
                if subst[p] != f:
                    return None
            else:
                subst[p] = f

        elif p != f:
            return None

    return subst


# Generate substitutions for all rule conditions
def match_all(conditions, facts, subst=None):

    if subst is None:
        subst = {}

    if not conditions:
        return [subst]

    results = []

    first = conditions[0]
    remaining = conditions[1:]

    for fact in facts:

        new_subst = unify(first, fact, subst.copy())

        if new_subst is not None:
            results.extend(
                match_all(remaining, facts, new_subst)
            )

    return results


# Apply substitution to generate a new fact
def apply_substitution(expression, subst):

    return tuple(
        subst.get(item, item) if item.startswith("?") else item
        for item in expression
    )


# Forward Chaining Algorithm
def forward_chaining(KB, rules):

    derived = True

    while derived:

        derived = False

        for conditions, conclusion in rules:

            substitutions = match_all(conditions, KB)

            for subst in substitutions:

                new_fact = apply_substitution(conclusion, subst)

                if new_fact not in KB:

                    KB.append(new_fact)

                    print("New Fact Derived:", new_fact)

                    derived = True

    return KB


# Execute Forward Chaining
print("Initial Knowledge Base:")
for fact in KB:
    print(fact)

print("\nForward Chaining Process:\n")

KB = forward_chaining(KB, rules)

# Check Goal
goal = ("Criminal", "Robert")

print("\nFinal Knowledge Base:")
for fact in KB:
    print(fact)

print("\nGoal:", goal)

if goal in KB:
    print("SUCCESS: Robert is a Criminal!")
else:
    print("FAILURE: Robert is not proven to be a Criminal.")

Initial Knowledge Base:
('American', 'Robert')
('Missile', 'T1')
('Owns', 'Robert', 'T1')
('Sells', 'Robert', 'T1', 'A')
('Enemy', 'America', 'A')

Forward Chaining Process:

New Fact Derived: ('Weapon', 'T1')
New Fact Derived: ('Hostile', 'A')
New Fact Derived: ('Criminal', 'Robert')

Final Knowledge Base:
('American', 'Robert')
('Missile', 'T1')
('Owns', 'Robert', 'T1')
('Sells', 'Robert', 'T1', 'A')
('Enemy', 'America', 'A')
('Weapon', 'T1')
('Hostile', 'A')
('Criminal', 'Robert')

Goal: ('Criminal', 'Robert')
SUCCESS: Robert is a Criminal!


In [2]:
class FOLForwardChaining:
    def __init__(self, initial_kb, rules):
        # Work with copies to avoid mutating shared global state unexpectedly
        self.kb = list(initial_kb)
        self.rules = rules

    def unify(self, pattern, fact, subst):
        if len(pattern) != len(fact):
            return None
        for p, f in zip(pattern, fact):
            if p.startswith("?"):
                if p in subst:
                    if subst[p] != f:
                        return None
                else:
                    subst[p] = f
            elif p != f:
                return None
        return subst

    def match_all(self, conditions, facts, subst=None):
        if subst is None:
            subst = {}
        if not conditions:
            return [subst]
        results = []
        first = conditions[0]
        remaining = conditions[1:]
        for fact in facts:
            new_subst = self.unify(first, fact, subst.copy())
            if new_subst is not None:
                results.extend(self.match_all(remaining, facts, new_subst))
        return results

    def apply_substitution(self, expression, subst):
        return tuple(
            subst.get(item, item) if item.startswith("?") else item
            for item in expression
        )

    def run(self, goal=None, verbose=True):
        derived = True
        if verbose:
            print("--- Starting Forward Chaining Inference ---")
        while derived:
            derived = False
            for conditions, conclusion in self.rules:
                substitutions = self.match_all(conditions, self.kb)
                for subst in substitutions:
                    new_fact = self.apply_substitution(conclusion, subst)
                    if new_fact not in self.kb:
                        self.kb.append(new_fact)
                        if verbose:
                            print(f"Derived New Fact: {new_fact}")
                        derived = True

        if verbose:
            print("--- Inference Complete ---\n")

        if goal is not None:
            success = goal in self.kb
            if verbose:
                if success:
                    print(f"SUCCESS: Goal {goal} is proven!")
                else:
                    print(f"FAILURE: Goal {goal} could not be proven.")
            return success, self.kb
        return self.kb

In [3]:
# Example Demonstration using a new/modified scenario:
# Proving "Colonel West is a criminal"

# 1. Define the generalized input data structures
custom_kb = [
    ("American", "West"),
    ("Missile", "M1"),
    ("Owns", "Nono", "M1"),
    ("Sells", "West", "M1", "Nono"),
    ("Enemy", "Nono", "America")
]

custom_rules = [
    # Rule 1: Missiles are weapons
    ([("Missile", "?x")], ("Weapon", "?x")),

    # Rule 2: Enemies of America are hostile
    ([("Enemy", "?x", "America")], ("Hostile", "?x")),

    # Rule 3: Americans selling weapons to hostile nations are criminals
    ([
        ("American", "?x"),
        ("Weapon", "?y"),
        ("Sells", "?x", "?y", "?z"),
        ("Hostile", "?z")
    ], ("Criminal", "?x"))
]

# 2. Instantiate the generalized solver
solver = FOLForwardChaining(initial_kb=custom_kb, rules=custom_rules)

# 3. Define your target goal
target_goal = ("Criminal", "West")

# 4. Execute inference
success, final_kb = solver.run(goal=target_goal, verbose=True)

--- Starting Forward Chaining Inference ---
Derived New Fact: ('Weapon', 'M1')
Derived New Fact: ('Hostile', 'Nono')
Derived New Fact: ('Criminal', 'West')
--- Inference Complete ---

SUCCESS: Goal ('Criminal', 'West') is proven!


### Interactive FOL Forward Chaining Solver
Run the cell below to dynamically input your own facts, rules, and goal.

In [4]:
import ast

def parse_tuple_input(prompt_text):
    """Helper to parse a user-inputted tuple or list of tuples."""
    val = input(prompt_text).strip()
    if not val:
        return []
    try:
        # Safely evaluate the literal structure entered by the user
        parsed = ast.literal_eval(val)
        return parsed
    except Exception as e:
        print(f"Error parsing input. Please ensure it is in valid Python format. Error: {e}")
        return None

def interactive_solver():
    print("=== Interactive FOL Forward Chaining Solver ===\n")

    # 1. Get Initial KB Facts
    print("Step 1: Enter your Initial Knowledge Base (Facts).")
    print("Example format: [(\"American\", \"Robert\"), (\"Missile\", \"T1\")]")
    kb_input = None
    while kb_input is None:
        kb_input = parse_tuple_input("Enter KB Facts: ")

    # 2. Get Rules
    print("\nStep 2: Enter your Inference Rules.")
    print("Example format: [ ([(\"Missile\", \"?x\")], (\"Weapon\", \"?x\")) ]")
    rules_input = None
    while rules_input is None:
        rules_input = parse_tuple_input("Enter Inference Rules: ")

    # 3. Get Target Goal
    print("\nStep 3: Enter your Target Goal.")
    print("Example format: (\"Criminal\", \"Robert\")")
    goal_input = None
    while goal_input is None:
        goal_input = parse_tuple_input("Enter Target Goal: ")
        if isinstance(goal_input, list) and len(goal_input) == 1:
            goal_input = goal_input[0]

    # 4. Instantiate and Run
    print("\nProcessing your input and running inference...")
    engine = FOLForwardChaining(initial_kb=kb_input, rules=rules_input)
    engine.run(goal=goal_input, verbose=True)

# Run the interactive interface
interactive_solver()

=== Interactive FOL Forward Chaining Solver ===

Step 1: Enter your Initial Knowledge Base (Facts).
Example format: [("American", "Robert"), ("Missile", "T1")]
Enter KB Facts: ("American", "Robert"),     ("Missile", "T1"),     ("Owns", "Robert", "T1"),     ("Sells", "Robert", "T1", "A"),     ("Enemy", "America", "A")

Step 2: Enter your Inference Rules.
Example format: [ ([("Missile", "?x")], ("Weapon", "?x")) ]
Enter Inference Rules: [("Missile", "?x")],      ("Weapon", "?x")),    ([("Enemy", "America", "?x")],      ("Hostile", "?x")), ([         ("American", "?x"),         ("Weapon", "?y"),         ("Sells", "?x", "?y", "?z"),         ("Hostile", "?z")     ],      ("Criminal", "?x")) ]
Error parsing input. Please ensure it is in valid Python format. Error: unmatched ')' (<unknown>, line 1)
Enter Inference Rules: [([("Missile", "?x")],      ("Weapon", "?x")),([("Enemy", "America", "?x")],      ("Hostile", "?x")),  ([         ("American", "?x"),         ("Weapon", "?y"),         ("Sell